# Colab Training Pipeline
This notebook runs your exact training pipeline from `train.py` natively in Colab cells. 
It **imports** your model architecture (`model.py`), dataset processing (`dataset.py`), and configs (`config.py`) so you don't have to redefine them.

**Setup Instructions:**
1. Upload your entire `Trans_trans` folder to Google Drive.
2. Run the cells below!

In [ ]:
# 1. Mount Google Drive and set up Python Path
from google.colab import drive
import sys
import os

drive.mount('/content/drive')

# Change this if your project is uploaded elsewhere in your Drive
PROJECT_DIR = '/content/drive/MyDrive/Trans_trans'
os.chdir(PROJECT_DIR)
print(f"Current working directory: {os.getcwd()}")

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

# Install requirements
!pip install -q torch torchvision torchaudio tokenizers datasets tensorboard tqdm

In [ ]:
# 2. Global Imports and Local Project Imports
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

from tokenizers import Tokenizer
from tokenizers.models import WordLevel 
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

from pathlib import Path
import json
from datasets import load_dataset
from tqdm import tqdm
import warnings

# --- IMPORTING FROM YOUR LOCAL FILES ---
from src.model.model import build_transformer
from src.utils.dataset import BilingualDataset, causal_mask
from src.utils.config import get_config, get_weights_file_path

warnings.filterwarnings("ignore")

In [ ]:
# 3. Validation and Translation Helpers
def greedy_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device):
    sos_idx = tokenizer_tgt.token_to_id("[SOS]")
    eos_idx = tokenizer_tgt.token_to_id("[EOS]")

    encoder_output = model.encode(source, source_mask)
    decoder_input = torch.empty(1, 1).fill_(sos_idx).type_as(source).to(device)
    while True:
        if decoder_input.size(1) == max_len:
            break
        decoder_mask = causal_mask(decoder_input.size(1)).type_as(source_mask).to(device)
        out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)
        prob = model.project(out[:, -1, :])
        _, next_token = torch.max(prob, dim=1)
        decoder_input = torch.cat([decoder_input, torch.empty(1, 1).type_as(source).fill_(next_token.item()).to(device)], dim=1)
        if next_token.item() == eos_idx:
            break

    return decoder_input.squeeze(0)

def run_validation(model, validation_ds, tokenizer_src, tokenizer_tgt, max_len, device, print_msg, global_state, writer, num_examples=2):
    model.eval()
    count = 0
    console_width = 80
    
    with torch.no_grad():
        for batch in validation_ds:
            count += 1
            encoder_input = batch['encoder_input'].to(device)
            encoder_mask = batch['encoder_mask'].to(device)
            
            assert encoder_input.size(0) == 1, "Batch size must be 1 for validation"
            model_output = greedy_decode(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, max_len, device)

            source_text = batch['src_text'][0]
            target_text = batch['tgt_text'][0]
            model_out_text = tokenizer_tgt.decode(model_output.detach().cpu().numpy())

            print_msg(f"\n{'='*console_width}")
            print_msg(f"Source: {source_text}")
            print_msg(f"Target: {target_text}")
            print_msg(f"Model Output: {model_out_text}")

            if count == num_examples:
                break

In [ ]:
# 4. Tokenization and Dataset Loaders
def get_all_sentences(ds, lang):
    for item in ds:
        yield item['translation'][lang]

def get_or_build_tokenizer(config, ds, lang):
    tokenizer_path = Path(config['tokenizer_file'].format(lang))

    if not Path.exists(tokenizer_path):
        tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
        tokenizer.pre_tokenizer = Whitespace()
        trainer = WordLevelTrainer(special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]", "[MASK]"], min_frequency=2)

        tokenizer.train_from_iterator(get_all_sentences(ds, lang), trainer=trainer)
        tokenizer.save(str(tokenizer_path))
    else:
        tokenizer = Tokenizer.from_file(str(tokenizer_path))
    return tokenizer

def get_ds(config):
    ds_raw = load_dataset('opus_books', f"{config['lang_src']}-{config['lang_tgt']}", split='train')

    tokenizer_src = get_or_build_tokenizer(config, ds_raw, config['lang_src'])
    tokenizer_tgt = get_or_build_tokenizer(config, ds_raw, config['lang_tgt'])

    train_ds_size = int(len(ds_raw) * 0.9)
    test_ds_size = len(ds_raw) - train_ds_size
    train_ds_raw, val_ds_raw = random_split(ds_raw, [train_ds_size, test_ds_size])

    train_ds = BilingualDataset(train_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])
    val_ds = BilingualDataset(val_ds_raw, tokenizer_src, tokenizer_tgt, config['lang_src'], config['lang_tgt'], config['seq_len'])

    max_len_src = 0
    max_len_tgt = 0

    for item in ds_raw:
        src_ids = tokenizer_src.encode(item['translation'][config['lang_src']]).ids
        tgt_ids = tokenizer_tgt.encode(item['translation'][config['lang_tgt']]).ids
        max_len_src = max(max_len_src, len(src_ids))
        max_len_tgt = max(max_len_tgt, len(tgt_ids))

    print(f"Max len src: {max_len_src}")
    print(f"Max len tgt: {max_len_tgt}")

    train_dataloader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    val_dataloader = DataLoader(val_ds, batch_size=1, shuffle=True)

    return train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt  

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model(config, vocab_src_len, vocab_tgt_len):
    model = build_transformer(vocab_src_len, vocab_tgt_len, config['seq_len'], config['seq_len'], config['d_model'])
    return model

In [ ]:
# 5. Initialize Model, Optimizer, and Config
config = get_config()

# Optional override for notebook experiments:
# config['batch_size'] = 16
# config['num_epochs'] = 30

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
Path(config['model_folder']).mkdir(parents=True, exist_ok=True)

train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

model_params = count_parameters(model)
print(f"\n{'='*50}")
print(f"Model Configuration:")
print(f"{'='*50}")
print(f"Total parameters: {model_params:,}")
print(f"Model dimension (d_model): {config['d_model']}")
print(f"Source vocab size: {tokenizer_src.get_vocab_size():,}")
print(f"Target vocab size: {tokenizer_tgt.get_vocab_size():,}")
print(f"Max sequence length: {config['seq_len']}")
print(f"Training batches: {len(train_dataloader):,}")
print(f"{'='*50}\n")

# Save Metadata
metadata = {
    "config": config,
    "model_parameters": int(model_params),
    "src_vocab_size": int(tokenizer_src.get_vocab_size()),
    "tgt_vocab_size": int(tokenizer_tgt.get_vocab_size()),
    "seq_len": int(config['seq_len']),
    "batch_size": int(config['batch_size']),
    "num_epochs": int(config['num_epochs']),
    "device": str(device),
}
metadata_path = Path(config['model_folder']) / "model_metadata.json"
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

writer = SummaryWriter(config['experiment_name'])
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], eps=1e-9)

initial_epoch = 0
global_step = 0
if config['preload'] is not None:
    model_filename = get_weights_file_path(config, config['preload'])
    print(f"Loading model from file: {model_filename}")
    state = torch.load(model_filename)
    initial_epoch = state['epoch'] + 1
    optimizer.load_state_dict(state['optimizer_state_dict'])
    model.load_state_dict(state['model_state_dict'])
    global_step = state['global_step']

loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer_src.token_to_id("[PAD]"), label_smoothing=0.1).to(device)

In [ ]:
# 6. Execute Training Loop
last_loss_value = None

for epoch in range(initial_epoch, config['num_epochs']):
    batch_iterator = tqdm(train_dataloader, desc=f"Processing epoch {epoch:02d}")
    for batch in batch_iterator:
        model.train()
        encoder_input = batch['encoder_input'].to(device)
        decoder_input = batch['decoder_input'].to(device)
        encoder_mask = batch['encoder_mask'].to(device)
        decoder_mask = batch['decoder_mask'].to(device)

        encoder_output = model.encode(encoder_input, encoder_mask)
        decoder_output = model.decode(encoder_output, encoder_mask, decoder_input, decoder_mask)
        proj_output = model.project(decoder_output)

        label = batch['label'].to(device)
        loss = loss_fn(proj_output.view(-1, tokenizer_tgt.get_vocab_size()), label.view(-1))    
        batch_iterator.set_postfix({f"loss": f"{loss.item():6.3f}"})

        last_loss_value = loss.item()

        writer.add_scalar("train loss", loss.item(), global_step)
        writer.flush()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        global_step += 1

    run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, config['seq_len'], device, lambda msg: batch_iterator.write(msg), global_step, writer)
    
    model_filename = get_weights_file_path(config, f'{epoch:02d}')
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'global_step': global_step
    }, model_filename)